In [15]:
import pandas as pd

data=pd.read_csv("Groceries data train.csv")
data = data.dropna()

# convert User_id, year, month, day, day_of_week to int
data['year'] = data['year'].astype('int')
data['month'] = data['month'].astype('int')
data['day'] = data['day'].astype('int')
data['day_of_week'] = data['day_of_week'].astype('int') 
data['User_id'] = data['User_id'].astype('int')
data.shape

(19382, 7)

In [24]:
# Group data by User_id and Date to form transactions
transactions = [a[1]['itemDescription'].tolist() for a in list(data.groupby(['User_id','Date']))]
print(f"Total number of transactions: {len(transactions)}")
print("\nSample transactions (first 2):")
for i, transaction in enumerate(transactions[:2]):
    print(f"Transaction {i+1}: {transaction}")

Total number of transactions: 8361

Sample transactions (first 2):
Transaction 1: ['whole milk', 'pastry', 'salty snack']
Transaction 2: ['sausage', 'whole milk', 'rolls/buns']


In [25]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)
df_encoded.head()

,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,bags,baking powder,bathroom cleaner,beef,berries,...,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [30]:
# Mining frequent itemsets using Apriori algorithm
from mlxtend.frequent_patterns import apriori, association_rules
# Set the minimum support threshold

min_support = 0.001
print(f"Using minimum support threshold: {min_support}")

# Apply the Apriori algorithm
frequent_itemsets = apriori(
    df_encoded, 
    min_support=min_support, 
    use_colnames=True, 
    verbose=1
)

print(f"\nNumber of frequent itemsets found: {len(frequent_itemsets)}")
print("\nTop 5 frequent itemsets by support:")
frequent_itemsets.sort_values(by='support', ascending=False).head(5)

Using minimum support threshold: 0.001
Processing 11706 combinations | Sampling itemset size 3

Number of frequent itemsets found: 462

Top 5 frequent itemsets by support:


,support,itemsets
145,0.130487,(whole milk)
88,0.106686,(other vegetables)
107,0.101902,(rolls/buns)
121,0.094606,(soda)
146,0.078579,(yogurt)


In [31]:
# Generating association rules
min_confidence = 0.001

rules = association_rules(
    frequent_itemsets, 
    metric="confidence", 
    min_threshold=min_confidence
)

print(f"\nNumber of association rules generated: {len(rules)}")
print("\nTop 5 rules by confidence:")
rules.sort_values(by='confidence', ascending=False).head(5)


Number of association rules generated: 628

Top 5 rules by confidence:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
308,(frankfurter),(whole milk),0.012199,0.130487,0.002033,0.166667,1.277269,1.0,0.000441,1.043416,0.219760,0.014456,0.041609,0.091124
285,(detergent),(rolls/buns),0.008492,0.101902,0.001196,0.140845,1.382166,1.0,0.000331,1.045328,0.278866,0.010953,0.043362,0.076291
527,(seasonal products),(rolls/buns),0.007774,0.101902,0.001076,0.138462,1.358776,1.0,0.000284,1.042436,0.266112,0.009912,0.040708,0.074512
158,(candy),(whole milk),0.016027,0.130487,0.002153,0.134328,1.029440,1.0,0.000062,1.004438,0.029064,0.014913,0.004418,0.075413
58,(bottled beer),(whole milk),0.040545,0.130487,0.005382,0.132743,1.017294,1.0,0.000091,1.002602,0.017718,0.032491,0.002595,0.086995


## Enhanced


In [33]:
# Calculate a custom score that combines confidence, lift, and support
def calculate_custom_score(rules_df):
    """
    This custom score balances:
    1. Confidence: reliability of the rule
    2. Lift: correlation strength between antecedent and consequent
    3. Support: prevalence of the pattern
    
    Formula: (0.5 * normalized_confidence) + (0.3 * normalized_lift) + (0.2 * normalized_support)
    """
    # Normalize metrics to 0-1 range
    norm_conf = rules_df['confidence'] / rules_df['confidence'].max()
    norm_lift = rules_df['lift'] / rules_df['lift'].max()
    norm_support = rules_df['support'] / rules_df['support'].max()
    
    # Calculate combined score
    custom_score = (0.5 * norm_conf) + (0.3 * norm_lift) + (0.2 * norm_support)
    
    return custom_score

rules['custom_score'] = calculate_custom_score(rules)
print("\nTop 5 rules by custom score:")
rules.sort_values(by='custom_score', ascending=False).head(5)


Top 5 rules by custom score:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,custom_score
308,(frankfurter),(whole milk),0.012199,0.130487,0.002033,0.166667,1.277269,1.0,0.000441,1.043416,0.219760,0.014456,0.041609,0.091124,0.730188
58,(bottled beer),(whole milk),0.040545,0.130487,0.005382,0.132743,1.017294,1.0,0.000091,1.002602,0.017718,0.032491,0.002595,0.086995,0.663280
285,(detergent),(rolls/buns),0.008492,0.101902,0.001196,0.140845,1.382166,1.0,0.000331,1.045328,0.278866,0.010953,0.043362,0.076291,0.649820
515,(pot plants),(yogurt),0.008372,0.078579,0.001076,0.128571,1.636204,1.0,0.000419,1.057368,0.392112,0.012535,0.054256,0.071135,0.647401
527,(seasonal products),(rolls/buns),0.007774,0.101902,0.001076,0.138462,1.358776,1.0,0.000284,1.042436,0.266112,0.009912,0.040708,0.074512,0.636665


In [35]:
def mine_patterns_for_user(user_id, min_support=0.001, min_confidence=0.001):
    """
    Mine patterns specifically for a given user.
    
    Parameters:
    -----------
    user_id : int
        The ID of the user to mine patterns for
    min_support : float
        Minimum support threshold for frequent itemsets
    min_confidence : float
        Minimum confidence threshold for association rules
        
    Returns:
    --------
    tuple
        (frequent_itemsets, association_rules) for the specified user
    """
    # Filter data for the specific user
    user_data = data[data['User_id'] == user_id]
    
    if len(user_data) == 0:
        print(f"No transactions found for User ID {user_id}")
        return None, None
    
    # Create transactions for this user
    user_transactions = [items['itemDescription'].tolist() 
                         for _, items in user_data.groupby('Date')]
    
    print(f"Found {len(user_transactions)} transactions for User ID {user_id}")
    
    # Check if user has enough transactions
    if len(user_transactions) < 2:
        print(f"User {user_id} has insufficient transaction history for pattern mining")
        return None, None
    
    # One-hot encode transactions
    te_user = TransactionEncoder()
    te_user_array = te_user.fit(user_transactions).transform(user_transactions)
    df_user_encoded = pd.DataFrame(te_user_array, columns=te_user.columns_)
    
    # Mine frequent itemsets
    user_frequent_itemsets = apriori(
        df_user_encoded, 
        min_support=min_support, 
        use_colnames=True
    )
    
    # Check if any frequent itemsets were found
    if len(user_frequent_itemsets) == 0:
        print(f"No frequent itemsets found for User ID {user_id}")
        return None, None
    
    # Generate association rules
    user_rules = association_rules(
        user_frequent_itemsets, 
        metric="confidence", 
        min_threshold=min_confidence
    )
    
    # Add custom score
    if len(user_rules) > 0:
        user_rules['custom_score'] = calculate_custom_score(user_rules)
    
    print(f"Generated {len(user_rules)} association rules for User ID {user_id}")
    
    return user_frequent_itemsets, user_rules

In [37]:
sample_user_id = 2075
user_itemsets, user_rules = mine_patterns_for_user(sample_user_id)

print(f"\nFrequent itemsets for User ID {sample_user_id}:")
user_rules.sort_values(by='custom_score', ascending=False).head(5)

Found 3 transactions for User ID 2075
Generated 6 association rules for User ID 2075

Frequent itemsets for User ID 2075:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,custom_score
0,(root vegetables),(baking powder),0.333333,0.333333,0.333333,1.0,3.0,1.0,0.222222,inf,1.0,1.0,1.0,1.0,1.0
1,(baking powder),(root vegetables),0.333333,0.333333,0.333333,1.0,3.0,1.0,0.222222,inf,1.0,1.0,1.0,1.0,1.0
2,(rolls/buns),(citrus fruit),0.333333,0.333333,0.333333,1.0,3.0,1.0,0.222222,inf,1.0,1.0,1.0,1.0,1.0
3,(citrus fruit),(rolls/buns),0.333333,0.333333,0.333333,1.0,3.0,1.0,0.222222,inf,1.0,1.0,1.0,1.0,1.0
4,(sausage),(soda),0.333333,0.333333,0.333333,1.0,3.0,1.0,0.222222,inf,1.0,1.0,1.0,1.0,1.0


In [42]:
def calculate_recency_score(user_id, rules_df):
    """
    Calculate a recency score for rules based on how recently the items were purchased.
    Higher score for more recent purchases.
    """
    user_data = data[data['User_id'] == user_id]
    
    if len(user_data) == 0:
        return pd.Series([0.5] * len(rules_df))
    
    max_date = user_data['Date'].max()
    recency_scores = []
    
    for _, rule in rules_df.iterrows():
        # Get items in the antecedent
        antecedent_items = list(rule['antecedents'])
        
        # Find the most recent date for each item in the antecedent
        item_dates = []
        for item in antecedent_items:
            item_transactions = user_data[user_data['itemDescription'] == item]
            if not item_transactions.empty:
                most_recent = item_transactions['Date'].max()
                days_since = (max_date - most_recent).days
                item_dates.append(days_since)
        
        if item_dates:
            # Calculate recency score (inversely proportional to days since last purchase)
            avg_days = sum(item_dates) / len(item_dates)
            # Use exponential decay: e^(-days/30) to map days to 0-1 range
            recency_score = np.exp(-avg_days/30)
            recency_scores.append(recency_score)
        else:
            # If no date information, assign neutral score
            recency_scores.append(0.5)
    
    return pd.Series(recency_scores)

# Apply recency score to the user's rules
user_rules['recency_score'] = calculate_recency_score(sample_user_id, user_rules)

# Combine scores (70% custom score, 30% recency)
user_rules['final_score'] = 0.7 * user_rules['custom_score'] + 0.3 * user_rules['recency_score']

print("\nTop 5 rules with recency consideration:")
user_rules.sort_values(by='final_score', ascending=False).head(5)


Top 5 rules with recency consideration:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,custom_score,recency_score,final_score
0,(root vegetables),(baking powder),0.333333,0.333333,0.333333,1.0,3.0,1.0,0.222222,inf,1.0,1.0,1.0,1.0,1.0,1.000000,1.000000
1,(baking powder),(root vegetables),0.333333,0.333333,0.333333,1.0,3.0,1.0,0.222222,inf,1.0,1.0,1.0,1.0,1.0,1.000000,1.000000
4,(sausage),(soda),0.333333,0.333333,0.333333,1.0,3.0,1.0,0.222222,inf,1.0,1.0,1.0,1.0,1.0,0.000484,0.700145
5,(soda),(sausage),0.333333,0.333333,0.333333,1.0,3.0,1.0,0.222222,inf,1.0,1.0,1.0,1.0,1.0,0.000484,0.700145
2,(rolls/buns),(citrus fruit),0.333333,0.333333,0.333333,1.0,3.0,1.0,0.222222,inf,1.0,1.0,1.0,1.0,1.0,0.000023,0.700007
